# Lab 1 - PyTorch Foundations: Logistic Regression and an MLP Classifier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hyperscaleailabs/ml-platform-engineering/blob/main/notebooks/01_pytorch_lr_mlp.ipynb)

**Runtime:** ~5 minutes on CPU. No GPU required, no downloads required.

This is the ground floor of the lab series. Everything in Labs 2-4 (attention, LoRA,
distributed training) is built out of the four primitives you will implement here:

1. **Tensors** - the data container, and why shapes/dtypes/devices are the source of most bugs.
2. **Autograd** - how PyTorch computes gradients, verified against gradients you derive by hand.
3. **A training loop** - forward, loss, backward, step. Written manually first, then with `nn.Module` + `optim`.
4. **Evaluation** - the difference between a loss going down and a model being good.

## What you will build

| Model | Dataset | Point of the exercise |
|---|---|---|
| Logistic regression, manual gradients | 2-D `make_moons` | Autograd is not magic - it matches calculus |
| Logistic regression, `nn.Module` | 2-D `make_moons` | A linear boundary cannot solve a non-linear problem |
| MLP | 2-D `make_moons` | One hidden layer buys you non-linearity |
| MLP | 8x8 digits (10-class) | The same loop scales to a real multiclass task |

## How to run this

* **Colab:** click the badge above. Runtime -> Run all. Nothing to install.
* **Locally:** `pip install torch numpy matplotlib scikit-learn` then `jupyter lab`.

Every section ends with a `check(...)` assertion. If a check fires, the cell above it is wrong -
that is the point, the notebook is self-verifying rather than asking you to eyeball plots.

## 0. Setup

Config is read from environment variables so this notebook can also be executed
non-interactively in CI (`LAB_EPOCHS=1 jupyter nbconvert --execute ...`). That is a habit worth
forming early: a notebook that only runs when a human clicks through it is not a reproducible artifact.

In [ ]:
import os
import sys
import math
import time
import platform

IN_COLAB = "google.colab" in sys.modules

# Colab ships torch/sklearn/matplotlib preinstalled, so this is a no-op there.
# Locally, install once: pip install torch numpy matplotlib scikit-learn
if IN_COLAB:
    print("Running in Google Colab - required packages are preinstalled.")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# ---- Config (override via environment variables) -------------------------------------------------
SEED = int(os.environ.get("LAB_SEED", 0))
EPOCHS = int(os.environ.get("LAB_EPOCHS", 200))          # for the 2-D toy problems
DIGIT_EPOCHS = int(os.environ.get("LAB_DIGIT_EPOCHS", 30))  # for the 10-class digits problem
BATCH_SIZE = int(os.environ.get("LAB_BATCH_SIZE", 64))
# --------------------------------------------------------------------------------------------------


def pick_device() -> torch.device:
    """CUDA (Colab GPU) > MPS (Apple silicon) > CPU.

    Everything in this notebook is small enough that CPU is fine - often faster, since
    kernel launch overhead dominates at these sizes. Device selection is here because you
    will need the exact same helper in Labs 2-4, where it stops being optional.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()


def set_seed(seed: int = SEED) -> None:
    """Seed every RNG this notebook touches.

    Note what this does *not* guarantee: cuDNN autotuning, atomics in scatter/reduce kernels, and
    multi-threaded CPU reductions can still make runs bit-wise different. Seeding buys you
    reproducible *sampling and initialization*, not bit-exact determinism. For that you also need
    `torch.use_deterministic_algorithms(True)` and to accept the performance hit.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

print(f"python      {platform.python_version()}")
print(f"torch       {torch.__version__}")
print(f"numpy       {np.__version__}")
print(f"device      {DEVICE}")


def check(condition: bool, message: str) -> None:
    """Tiny assertion helper so failures read like a test suite instead of a traceback."""
    if not condition:
        raise AssertionError(f"CHECK FAILED: {message}")
    print(f"  ok - {message}")

## 1. Tensors

A `torch.Tensor` is an n-dimensional array plus three pieces of metadata that cause most of the
bugs you will hit in practice:

* **shape** - silent broadcasting turns a shape bug into a wrong number, not a crash.
* **dtype** - `float32` is the default; `int64` for label/index tensors; `bfloat16` on modern accelerators.
* **device** - a tensor on `cpu` and one on `cuda` cannot be combined.

The cell below demonstrates the single most common shape bug in ML code.

In [ ]:
x = torch.arange(6, dtype=torch.float32).reshape(2, 3)
print("x =\n", x)
print(f"shape={tuple(x.shape)} dtype={x.dtype} device={x.device}")

# Broadcasting: shapes are aligned from the right, and size-1 dims are stretched.
row = torch.tensor([10.0, 20.0, 30.0])       # (3,)   -> broadcast over rows
col = torch.tensor([[100.0], [200.0]])       # (2, 1) -> broadcast over columns
print("\nx + row (adds per-column):\n", x + row)
print("\nx + col (adds per-row):\n", x + col)

# The classic bug: a (N,) prediction vector minus an (N,1) target silently becomes (N,N).
pred = torch.randn(4)
target = torch.randn(4, 1)
print(f"\npred{tuple(pred.shape)} - target{tuple(target.shape)} -> {tuple((pred - target).shape)}  <- 16 elements, not 4")
print(f"pred{tuple(pred.shape)} - target.squeeze(){tuple(target.squeeze(-1).shape)} -> {tuple((pred - target.squeeze(-1)).shape)}  <- correct")

check((pred - target).shape == (4, 4), "broadcasting a (N,) against a (N,1) silently produces (N,N)")
check((pred - target.squeeze(-1)).shape == (4,), "squeezing the trailing dim gives the intended elementwise result")

> **Habit worth building:** annotate tensor shapes in comments (`# (B, T, C)`) and assert them at
> function boundaries. Shape asserts are free at these sizes and they catch the class of bug that
> otherwise shows up as "loss plateaus and I do not know why" three hours later.

## 2. Autograd, checked against calculus

PyTorch builds a dynamic graph of operations as you execute them. `loss.backward()` walks that
graph in reverse, accumulating `d loss / d param` into each leaf tensor's `.grad`.

To convince yourself it is just the chain rule, we take **binary logistic regression** and derive
the gradient by hand.

Model: $z = Xw + b$, &nbsp; $\hat{y} = \sigma(z) = \frac{1}{1+e^{-z}}$

Loss (mean binary cross-entropy over $N$ examples):

$$\mathcal{L} = -\frac{1}{N}\sum_i \left[ y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i) \right]$$

The sigmoid and the log cancel beautifully, leaving one of the cleanest gradients in ML:

$$\frac{\partial \mathcal{L}}{\partial z_i} = \frac{\hat{y}_i - y_i}{N}
\quad\Longrightarrow\quad
\nabla_w \mathcal{L} = \frac{1}{N} X^\top (\hat{y} - y), \qquad
\nabla_b \mathcal{L} = \frac{1}{N}\sum_i (\hat{y}_i - y_i)$$

Let us verify that against `.backward()`.

In [ ]:
set_seed()

N, D = 200, 3
X_ = torch.randn(N, D)
y_ = (torch.rand(N) < 0.5).float()

w = torch.randn(D, requires_grad=True)   # leaf tensors: gradients accumulate into .grad
b = torch.zeros(1, requires_grad=True)

# Forward pass. Use the numerically stable fused loss rather than sigmoid() then log().
z = X_ @ w + b                                   # (N,)
loss = F.binary_cross_entropy_with_logits(z, y_)
loss.backward()

# Analytic gradients derived above.
with torch.no_grad():
    y_hat = torch.sigmoid(X_ @ w + b)
    grad_w_manual = X_.T @ (y_hat - y_) / N
    grad_b_manual = (y_hat - y_).mean().reshape(1)

print(f"loss                 {loss.item():.6f}")
print(f"autograd  dL/dw      {w.grad.numpy()}")
print(f"analytic  dL/dw      {grad_w_manual.numpy()}")
print(f"max |difference|     {(w.grad - grad_w_manual).abs().max().item():.3e}")

check(torch.allclose(w.grad, grad_w_manual, atol=1e-6), "autograd dL/dw matches the hand-derived gradient")
check(torch.allclose(b.grad, grad_b_manual, atol=1e-6), "autograd dL/db matches the hand-derived gradient")

### Why `binary_cross_entropy_with_logits` and not `sigmoid` + `log`

`log(sigmoid(z))` overflows to `-inf` for large negative `z`, because `sigmoid(z)` underflows to
exactly 0.0 in float32. The fused version uses the log-sum-exp trick internally and stays finite.
This is the same reason Labs 2-4 always feed *logits* into the loss, never probabilities.

In [ ]:
z_extreme = torch.tensor([-100.0, -40.0, 0.0, 40.0])
naive = torch.log(torch.sigmoid(z_extreme))
stable = F.logsigmoid(z_extreme)
print(f"naive  log(sigmoid(z)) : {naive.numpy()}")
print(f"stable logsigmoid(z)   : {stable.numpy()}")

check(torch.isinf(naive).any(), "the naive formulation produces -inf and would poison the gradient")
check(torch.isfinite(stable).all(), "the fused formulation stays finite")

### Gradient accumulation is a feature, not a bug

`.backward()` **adds** into `.grad`. That is what makes gradient accumulation across micro-batches
possible (critical when a batch does not fit in GPU memory - you will use it in Lab 3). The cost is
that forgetting `optimizer.zero_grad()` silently sums gradients across steps and wrecks training.

In [ ]:
w2 = torch.tensor([1.0], requires_grad=True)
for step in range(3):
    (w2 * 3.0).backward()
    print(f"after backward #{step + 1}: w2.grad = {w2.grad.item()}")   # 3, 6, 9 - accumulating

w2.grad = None   # equivalent to optimizer.zero_grad(set_to_none=True)
(w2 * 3.0).backward()
print(f"after zeroing then backward: w2.grad = {w2.grad.item()}")

check(w2.grad.item() == 3.0, "zeroing grads restores the single-step gradient")

## 3. The data: `make_moons`

Two interleaving half-circles. Deliberately **not** linearly separable, so it exposes the exact
limitation of logistic regression that the MLP will fix.

In [ ]:
set_seed()

X_np, y_np = make_moons(n_samples=1000, noise=0.20, random_state=SEED)
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np, y_np, test_size=0.25, random_state=SEED, stratify=y_np
)

# Standardize using TRAIN statistics only. Computing mean/std over the full dataset leaks test
# information into training - a small effect here, a career-limiting one on a real benchmark.
mu, sigma = X_train_np.mean(axis=0), X_train_np.std(axis=0)
X_train_np = (X_train_np - mu) / sigma
X_test_np = (X_test_np - mu) / sigma

X_train = torch.tensor(X_train_np, dtype=torch.float32, device=DEVICE)
y_train = torch.tensor(y_train_np, dtype=torch.float32, device=DEVICE)
X_test = torch.tensor(X_test_np, dtype=torch.float32, device=DEVICE)
y_test = torch.tensor(y_test_np, dtype=torch.float32, device=DEVICE)

print(f"train {tuple(X_train.shape)}  test {tuple(X_test.shape)}  positives in train: {y_train.mean():.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_train_np[:, 0], X_train_np[:, 1], c=y_train_np, cmap="coolwarm", s=12, edgecolors="none")
ax.set_title("make_moons (train split, standardized)")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
plt.tight_layout(); plt.show()

check(abs(X_train_np.mean()) < 1e-6, "train features are zero-mean after standardization")

## 4. Training loop v1 - written by hand

No `nn.Module`, no optimizer, no dataloader. Full-batch gradient descent with the gradients we
derived in section 2. Every framework abstraction you use later is a wrapper around these six lines.

In [ ]:
set_seed()

w = torch.zeros(2, device=DEVICE, requires_grad=True)
b = torch.zeros(1, device=DEVICE, requires_grad=True)
lr = 0.5
manual_history = []

for epoch in range(EPOCHS):
    # 1. forward
    logits = X_train @ w + b
    loss = F.binary_cross_entropy_with_logits(logits, y_train)

    # 2. backward
    loss.backward()

    # 3. update - inside no_grad, otherwise the update itself joins the autograd graph
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    # 4. reset for the next step
    w.grad = None
    b.grad = None

    manual_history.append(loss.item())

print(f"epoch   0 loss {manual_history[0]:.4f}")
print(f"epoch {EPOCHS - 1:3d} loss {manual_history[-1]:.4f}")

with torch.no_grad():
    manual_test_acc = (((X_test @ w + b) > 0).float() == y_test).float().mean().item()
print(f"test accuracy {manual_test_acc:.3f}")

check(manual_history[-1] < manual_history[0], "loss decreased")
check(manual_test_acc > 0.80, f"hand-written logistic regression beats 80% accuracy (got {manual_test_acc:.3f})")

## 5. Training loop v2 - `nn.Module` + `optim`

Same math, idiomatic PyTorch. What the framework buys you:

* `nn.Module` tracks parameters (`.parameters()`), device movement (`.to()`), and train/eval mode.
* `optim.*` implements update rules more sophisticated than plain SGD - Adam adapts a per-parameter
  step size from running estimates of the gradient's first and second moments.
* `DataLoader` handles shuffling, batching, and (with `num_workers>0`) prefetching.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


class LogisticRegression(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.linear = nn.Linear(in_features, 1)

    def forward(self, x):                 # (B, in_features) -> (B,)
        return self.linear(x).squeeze(-1)  # return LOGITS, never probabilities


def train_binary(model, X_tr, y_tr, X_te, y_te, epochs=EPOCHS, lr=0.05, batch_size=BATCH_SIZE, log_every=None):
    """One reusable loop for every binary model in this notebook.

    Returns a history dict. Notice the two things people forget: `model.train()` /
    `model.eval()` (which toggle dropout and batchnorm), and wrapping evaluation in
    `torch.no_grad()` (which stops autograd from building a graph you will never backprop through).
    """
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    hist = {"train_loss": [], "test_loss": [], "test_acc": []}

    for epoch in range(epochs):
        model.train()
        running, seen = 0.0, 0
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            loss = F.binary_cross_entropy_with_logits(model(xb), yb)
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
            seen += xb.size(0)

        model.eval()
        with torch.no_grad():
            test_logits = model(X_te)
            test_loss = F.binary_cross_entropy_with_logits(test_logits, y_te).item()
            test_acc = (((test_logits > 0).float()) == y_te).float().mean().item()

        hist["train_loss"].append(running / seen)
        hist["test_loss"].append(test_loss)
        hist["test_acc"].append(test_acc)

        if log_every and (epoch % log_every == 0 or epoch == epochs - 1):
            print(f"epoch {epoch:4d}  train {hist['train_loss'][-1]:.4f}  test {test_loss:.4f}  acc {test_acc:.3f}")

    return hist


set_seed()
lr_model = LogisticRegression(2)
n_params = sum(p.numel() for p in lr_model.parameters())
print(f"logistic regression: {n_params} trainable parameters (w1, w2, b)\n")

lr_hist = train_binary(lr_model, X_train, y_train, X_test, y_test, log_every=max(1, EPOCHS // 5))
lr_acc = lr_hist["test_acc"][-1]

check(n_params == 3, "logistic regression on 2-D input has exactly 3 parameters")
check(lr_acc > 0.80, f"logistic regression reaches >80% (got {lr_acc:.3f})")
check(lr_acc < 0.95, f"...but stalls below 95% - a line cannot carve out two moons (got {lr_acc:.3f})")

That last check is the interesting one. The model is **not undertrained** - it has converged. It is
*underspecified*: the hypothesis class (straight lines) does not contain a good solution. More
epochs, more data, and a better optimizer all fail to help. You need a different function class.

## 6. The MLP

An MLP stacks affine maps with a non-linearity between them:

$$h = \phi(W_1 x + b_1), \qquad z = W_2 h + b_2$$

The non-linearity $\phi$ is doing all of the work. Without it, $W_2(W_1x + b_1) + b_2$ collapses
into a single affine map - stacking linear layers gives you back logistic regression with extra
steps. We prove that below rather than asserting it.

**Why ReLU** ($\phi(x)=\max(0,x)$): its gradient is exactly 1 on the active half, so it does not
saturate the way sigmoid/tanh do, and it is one comparison to evaluate. Its failure mode is dead
units (a unit whose pre-activation is always negative gets zero gradient forever), which is what
GELU and SiLU/Swish - the activations used in Labs 2-4 - smooth away.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features: int, hidden: int = 32, out_features: int = 1,
                 depth: int = 2, activation: str = "relu", dropout: float = 0.0):
        super().__init__()
        act = {"relu": nn.ReLU, "tanh": nn.Tanh, "gelu": nn.GELU, "identity": nn.Identity}[activation]
        layers, d = [], in_features
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), act()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = hidden
        layers.append(nn.Linear(d, out_features))
        self.net = nn.Sequential(*layers)
        self.out_features = out_features

    def forward(self, x):
        out = self.net(x)
        return out.squeeze(-1) if self.out_features == 1 else out


# Proof that a "deep" network with no activation is still linear: collapse it into one matrix.
set_seed()
linear_stack = MLP(2, hidden=32, depth=3, activation="identity").to(DEVICE)
probe = torch.randn(64, 2, device=DEVICE)
with torch.no_grad():
    W_eff = torch.autograd.functional.jacobian(lambda v: linear_stack(v.unsqueeze(0)).squeeze(0),
                                               torch.zeros(2, device=DEVICE))
    b_eff = linear_stack(torch.zeros(1, 2, device=DEVICE))
    collapsed = probe @ W_eff + b_eff
    actual = linear_stack(probe)
print(f"3 hidden layers, no activation -> equivalent to one line: max deviation {(collapsed - actual).abs().max():.2e}")
check(torch.allclose(collapsed, actual, atol=1e-4), "a stack of Linear layers with no activation is exactly affine")

In [ ]:
set_seed()
mlp = MLP(2, hidden=32, depth=2, activation="relu").to(DEVICE)
print(f"MLP: {sum(p.numel() for p in mlp.parameters())} trainable parameters\n")

mlp_hist = train_binary(mlp, X_train, y_train, X_test, y_test, log_every=max(1, EPOCHS // 5))
mlp_acc = mlp_hist["test_acc"][-1]

check(mlp_acc > lr_acc, f"MLP ({mlp_acc:.3f}) beats logistic regression ({lr_acc:.3f})")
check(mlp_acc > 0.95, f"MLP clears 95% on make_moons (got {mlp_acc:.3f})")

### Look at the decision boundaries

This is why the toy 2-D problem is worth the time: you can *see* the hypothesis class. Logistic
regression can only draw a straight line. The MLP draws a piecewise-linear boundary - one linear
piece per active ReLU region - which at 32 hidden units is enough to trace the moons.

In [ ]:
def plot_decision_boundary(models: dict, X_np_, y_np_, title=""):
    x1 = np.linspace(X_np_[:, 0].min() - 0.5, X_np_[:, 0].max() + 0.5, 300)
    x2 = np.linspace(X_np_[:, 1].min() - 0.5, X_np_[:, 1].max() + 0.5, 300)
    xx, yy = np.meshgrid(x1, x2)
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32, device=DEVICE)

    fig, axes = plt.subplots(1, len(models), figsize=(5.2 * len(models), 4.2), squeeze=False)
    for ax, (name, model) in zip(axes[0], models.items()):
        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model(grid)).cpu().numpy().reshape(xx.shape)
        ax.contourf(xx, yy, probs, levels=25, cmap="coolwarm", alpha=0.75)
        ax.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=1.5)
        ax.scatter(X_np_[:, 0], X_np_[:, 1], c=y_np_, cmap="coolwarm", s=10, edgecolors="k", linewidths=0.3)
        ax.set_title(name)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title)
    plt.tight_layout(); plt.show()


plot_decision_boundary(
    {f"Logistic regression - acc {lr_acc:.3f}": lr_model, f"MLP (32 hidden) - acc {mlp_acc:.3f}": mlp},
    X_test_np, y_test_np,
    title="Black line = the 0.5 decision boundary",
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lr_hist["train_loss"], label="LR train")
axes[0].plot(lr_hist["test_loss"], label="LR test", ls="--")
axes[0].plot(mlp_hist["train_loss"], label="MLP train")
axes[0].plot(mlp_hist["test_loss"], label="MLP test", ls="--")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss"); axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(lr_hist["test_acc"], label="LR")
axes[1].plot(mlp_hist["test_acc"], label="MLP")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("test accuracy"); axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Capacity and overfitting

More capacity is not free. Sweeping hidden width shows the classic bias/variance shape: too small
and the model cannot represent the boundary, too large and it starts memorizing the 20% label noise
we injected into the data. Watch the **gap** between train and test loss, not just test accuracy.

In [ ]:
widths = [1, 2, 4, 16, 64, 256]
sweep = []
for h in widths:
    set_seed()
    m = MLP(2, hidden=h, depth=2).to(DEVICE)
    hh = train_binary(m, X_train, y_train, X_test, y_test, epochs=max(30, EPOCHS // 2))
    sweep.append({
        "hidden": h,
        "params": sum(p.numel() for p in m.parameters()),
        "train_loss": hh["train_loss"][-1],
        "test_loss": hh["test_loss"][-1],
        "test_acc": hh["test_acc"][-1],
    })

print(f"{'hidden':>7} {'params':>7} {'train':>8} {'test':>8} {'gap':>8} {'acc':>7}")
for r in sweep:
    print(f"{r['hidden']:>7} {r['params']:>7} {r['train_loss']:>8.4f} {r['test_loss']:>8.4f} "
          f"{r['test_loss'] - r['train_loss']:>8.4f} {r['test_acc']:>7.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([r["hidden"] for r in sweep], [r["train_loss"] for r in sweep], "o-", label="train loss")
ax.plot([r["hidden"] for r in sweep], [r["test_loss"] for r in sweep], "o--", label="test loss")
ax.set_xscale("log", base=2); ax.set_xlabel("hidden units"); ax.set_ylabel("BCE loss")
ax.set_title("Capacity sweep - the train/test gap is the overfitting signal")
ax.legend(); plt.tight_layout(); plt.show()

check(sweep[0]["test_acc"] < sweep[3]["test_acc"], "a 1-unit hidden layer underfits relative to 16 units")
check(sweep[-1]["train_loss"] < sweep[0]["train_loss"], "the widest model fits the training set best")

## 8. Multiclass - 8x8 handwritten digits

Same loop, three changes, all of which trip people up:

| Binary | Multiclass |
|---|---|
| output shape `(B,)` | output shape `(B, C)` |
| `binary_cross_entropy_with_logits` | `cross_entropy` (log-softmax + NLL, fused) |
| labels `float32` in {0.,1.} | labels `int64` class indices in [0, C) |

`cross_entropy` expects **raw logits**. Passing it softmax probabilities is a real and common bug:
it does not error, it just applies softmax twice, flattening your gradients and making training
mysteriously slow.

`load_digits` is bundled with scikit-learn, so this section works with no network access.

In [ ]:
digits = load_digits()
Xd, yd = digits.data.astype(np.float32) / 16.0, digits.target.astype(np.int64)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=SEED, stratify=yd)

Xd_train = torch.tensor(Xd_tr, device=DEVICE)
yd_train = torch.tensor(yd_tr, device=DEVICE)
Xd_test = torch.tensor(Xd_te, device=DEVICE)
yd_test = torch.tensor(yd_te, device=DEVICE)

print(f"train {tuple(Xd_train.shape)}  test {tuple(Xd_test.shape)}  classes {len(np.unique(yd))}")

fig, axes = plt.subplots(1, 10, figsize=(11, 1.5))
for c, ax in enumerate(axes):
    ax.imshow(digits.images[np.where(digits.target == c)[0][0]], cmap="gray_r")
    ax.set_title(str(c)); ax.axis("off")
plt.suptitle("One example per class (8x8, 17 grey levels)"); plt.tight_layout(); plt.show()

In [ ]:
def train_multiclass(model, X_tr, y_tr, X_te, y_te, epochs=DIGIT_EPOCHS, lr=1e-3,
                     batch_size=BATCH_SIZE, weight_decay=0.0, log_every=10):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    hist = {"train_loss": [], "test_loss": [], "test_acc": []}

    for epoch in range(epochs):
        model.train()
        running, seen = 0.0, 0
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(xb), yb)     # logits in, int64 labels in
            loss.backward()
            # Clip the global gradient norm. Unnecessary here, essential in Labs 2-4 where a single
            # bad batch can produce a gradient spike that destroys an otherwise healthy run.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            running += loss.item() * xb.size(0)
            seen += xb.size(0)

        model.eval()
        with torch.no_grad():
            logits = model(X_te)
            hist["test_loss"].append(F.cross_entropy(logits, y_te).item())
            hist["test_acc"].append((logits.argmax(-1) == y_te).float().mean().item())
        hist["train_loss"].append(running / seen)

        if log_every and (epoch % log_every == 0 or epoch == epochs - 1):
            print(f"epoch {epoch:3d}  train {hist['train_loss'][-1]:.4f}  "
                  f"test {hist['test_loss'][-1]:.4f}  acc {hist['test_acc'][-1]:.4f}")
    return hist


set_seed()
digit_lr = MLP(64, hidden=1, out_features=10, depth=0)         # depth=0 -> a bare Linear(64,10)
print("softmax regression (no hidden layer):", sum(p.numel() for p in digit_lr.parameters()), "params")
digit_lr_hist = train_multiclass(digit_lr, Xd_train, yd_train, Xd_test, yd_test, lr=5e-3)

set_seed()
digit_mlp = MLP(64, hidden=128, out_features=10, depth=2, activation="gelu", dropout=0.1)
print("\nMLP 64-128-128-10:", sum(p.numel() for p in digit_mlp.parameters()), "params")
digit_mlp_hist = train_multiclass(digit_mlp, Xd_train, yd_train, Xd_test, yd_test, lr=3e-3)

lin_acc, mlp10_acc = digit_lr_hist["test_acc"][-1], digit_mlp_hist["test_acc"][-1]
print(f"\nlinear {lin_acc:.4f}   MLP {mlp10_acc:.4f}   (+{mlp10_acc - lin_acc:.4f})")
check(lin_acc > 0.90, f"even a linear model does well on 8x8 digits (got {lin_acc:.4f})")
check(mlp10_acc > lin_acc, f"the MLP is better (got {mlp10_acc:.4f})")

Note how *small* the MLP's advantage is here - a couple of percentage points for 30x the
parameters. That is the honest result, and it is worth internalizing: 8x8 digits are close to
linearly separable in pixel space, so the extra capacity has little to bite on. `make_moons` was
constructed so the gap would be dramatic. Real datasets rarely are, and "add a bigger model" is
frequently the wrong answer. Always run the linear baseline first.

### Accuracy is not enough

97% accuracy on 10 balanced classes still means ~13 mistakes. A confusion matrix tells you
*which* ones, and that is what drives the next iteration - here the errors concentrate on digit
pairs that genuinely look alike at 8x8 resolution (1/8, 3/9, 5/9).

In [ ]:
digit_mlp.eval()
with torch.no_grad():
    pred = digit_mlp(Xd_test).argmax(-1).cpu().numpy()
cm = confusion_matrix(yd_te, pred)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_title(f"Confusion matrix - accuracy {mlp10_acc:.4f}")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

errors = np.where(pred != yd_te)[0]
print(f"{len(errors)} misclassified out of {len(yd_te)}")
if len(errors):
    show = errors[:8]
    fig, axes = plt.subplots(1, len(show), figsize=(1.4 * len(show), 1.8))
    for idx, ax in zip(show, np.atleast_1d(axes)):
        ax.imshow(Xd_te[idx].reshape(8, 8), cmap="gray_r")
        ax.set_title(f"{yd_te[idx]}->{pred[idx]}", fontsize=9); ax.axis("off")
    plt.suptitle("Every mistake the model made (first 8)"); plt.tight_layout(); plt.show()

check(cm.sum() == len(yd_te), "the confusion matrix accounts for every test example")
# float32 accumulation on the torch side vs float64 on the numpy side - compare with a tolerance,
# never with ==. This bites people constantly when reconciling metrics across two libraries.
check(abs(np.trace(cm) / cm.sum() - mlp10_acc) < 1e-6, "the trace over the total is the accuracy")

## 9. Save and reload

Save the `state_dict` (a plain dict of tensors), never the pickled module - pickled modules break
the moment you refactor the class. `weights_only=True` is the safe default when loading anything
you did not produce yourself.

In [ ]:
ckpt_path = "mlp_digits.pt"
torch.save({"state_dict": digit_mlp.state_dict(),
            "config": {"in_features": 64, "hidden": 128, "out_features": 10, "depth": 2,
                       "activation": "gelu", "dropout": 0.1},
            "test_acc": mlp10_acc}, ckpt_path)

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
restored = MLP(**ckpt["config"]).to(DEVICE)
restored.load_state_dict(ckpt["state_dict"])
restored.eval()
with torch.no_grad():
    restored_acc = (restored(Xd_test).argmax(-1) == yd_test).float().mean().item()

print(f"checkpoint {os.path.getsize(ckpt_path) / 1024:.1f} KiB   reloaded accuracy {restored_acc:.4f}")
check(restored_acc == mlp10_acc, "the reloaded model reproduces the original accuracy exactly")
os.remove(ckpt_path)

## 10. Exercises

Work through these before moving on to Lab 2 - the transformer notebook assumes you are fluent
with the training loop above.

1. **Weight decay.** Re-run the width sweep with `weight_decay=1e-2`. Does the train/test gap at
   `hidden=256` close? Which is the better regularizer here, weight decay or dropout?
2. **Learning rate.** Train the MLP at `lr` in `{1e-4, 1e-3, 1e-2, 1e-1, 1.0}`, plot loss curves on
   one axis, and identify the divergence threshold. Then add a cosine schedule
   (`torch.optim.lr_scheduler.CosineAnnealingLR`) and see whether the best fixed rate is beaten.
3. **Dead ReLUs.** Register a forward hook on the first activation and count units that output 0
   for every test example. Does the count grow with learning rate? Does GELU fix it?
4. **Class imbalance.** Subsample the `make_moons` positives to 5% of the data. Accuracy will look
   great and the model will be useless - fix it with `pos_weight` in
   `binary_cross_entropy_with_logits`, and report balanced accuracy instead.
5. **Initialization.** Replace `nn.Linear`'s default init with all-zeros. Explain, using the
   gradient you derived in section 2, why the hidden units then stay identical forever.

A worked solution to #3 is below, since forward hooks are the standard tool for the activation
statistics you will want when a real training run goes wrong.

In [ ]:
# Solution sketch for exercise 3: count dead ReLU units with a forward hook.
set_seed()
probe_model = MLP(2, hidden=32, depth=2, activation="relu").to(DEVICE)
train_binary(probe_model, X_train, y_train, X_test, y_test, epochs=max(20, EPOCHS // 4), lr=0.05)

activations = {}
handles = [
    m.register_forward_hook(lambda mod, inp, out, i=i: activations.__setitem__(i, out.detach()))
    for i, m in enumerate(probe_model.net) if isinstance(m, nn.ReLU)
]
probe_model.eval()
with torch.no_grad():
    probe_model(X_test)
for h in handles:
    h.remove()

for layer_idx, act in activations.items():
    alive = (act > 0).any(dim=0)
    print(f"ReLU layer {layer_idx}: {int(alive.sum())}/{act.shape[1]} units fire on at least one "
          f"test example, mean activation rate {(act > 0).float().mean():.3f}")

check(len(activations) == 2, "the hook captured both ReLU layers")

## What you built, and what comes next

* Verified autograd against a hand-derived gradient - the chain rule, nothing more.
* Wrote the training loop by hand, then in idiomatic PyTorch, and saw why `zero_grad`,
  `no_grad`, `train()/eval()`, and logits-not-probabilities all matter.
* Watched a model fail for a structural reason (wrong hypothesis class), not a tuning reason.
* Scaled the identical loop to a 10-class problem and read the errors, not just the metric.

**Lab 2** keeps this exact training loop and swaps the MLP for a transformer: you will implement
scaled dot-product attention, RoPE, RMSNorm, and SwiGLU from scratch, verify each against the
PyTorch reference, and train a small language model. Those are the components of the Qwen model
you will fine-tune with LoRA in Labs 3 and 4.